In [1]:
# -*- coding: utf-8 -*-
import ee, geemap, pandas as pd, re, time

# ====== 用户配置 ======
CSV_PATH       = r"E:\NWP\CS2_S1_matched\time_match_2015.csv"
SCENENAME_COL  = "sceneName"         # 你的 CSV 场景名列
DRIVE_FOLDER   = "S1_EW_exports"     # 统一的 Google Drive 目标文件夹（不会再建子文件夹）
SCALE_M        = 40                  # EW 建议 40m；仍嫌大可改 60/80
TARGET_BANDS   = ["HH", "HV", "angle"]        # 只要 HH 就改成 ["HH"]；只要 HV 就 ["HV"]
MAX_EXPORTS    = 500                 # 一次提交的最大任务数
SLEEP_SEC      = 0.3                 # 任务提交间隔，防止配额触发
USE_AOI        = False               # True时使用自定义AOI裁剪
# ======================

In [2]:
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

In [3]:
# 读取CSV -> 去重的 sceneName 列表
df = pd.read_csv(CSV_PATH)
scenes = (df[SCENENAME_COL].astype(str).str.strip().dropna().unique().tolist())
print(f"[INFO] scenes: {len(scenes)}")

scenes = scenes[:30]

[INFO] scenes: 773


In [5]:
def get_s1(scene):
    col = (ee.ImageCollection("COPERNICUS/S1_GRD")
           .filter(ee.Filter.eq("system:index", scene))
           .filter(ee.Filter.eq("productType", "GRD"))
           .filter(ee.Filter.eq("instrumentMode", "EW")))
    img = col.first()
    return ee.Image(img) if img else None

def available_target_bands(img, target):
    # 读取影像内实际存在的波段，与目标交集
    names = ee.List(img.bandNames())
    out = []
    for b in target:
        try:
            if names.contains(b).getInfo():
                out.append(b)
        except Exception:
            pass
    return out

def to_int16_db100(img_db, bands):
    # 将 dB×100 转 int16，显著减小体积；保留掩膜
    return img_db.select(bands).multiply(100).toInt16()

launched = 0
for i, scene in enumerate(scenes, 1):
    print(f"\n[{i}/{len(scenes)}] {scene}")
    img = get_s1(scene)
    if img is None:
        print("  [MISS] scene not found in GEE")
        continue

    bands = available_target_bands(img, TARGET_BANDS)
    if not bands:
        print("  [MISS] no requested bands (maybe VV/VH only)")
        continue

    # 区域：整景或 AOI
    region = img.geometry()

    # 压缩为 int16（dB×100）
    small_img = to_int16_db100(img, bands)

    # —— 关键命名策略 —— 
    # 统一 folder=DRIVE_FOLDER；每个任务用唯一的 fileNamePrefix（文件名），
    # description 只是任务名，不会在 Drive 再建子文件夹。
    file_prefix = f"{scene}_EW_" + "_".join(bands) + "_int16x100"
    desc        = f"export_{file_prefix}"   # 任务名（不影响 Drive 路径）

    if launched >= MAX_EXPORTS:
        print("  [STOP] reach MAX_EXPORTS; split into batches if needed.")
        break

    ee.batch.Export.image.toDrive(
        image         = small_img,
        description   = desc,               # 任务名
        # folder        = DRIVE_FOLDER,       # 统一父目录
        fileNamePrefix= file_prefix,        # 直接作为文件名；不会建子文件夹
        region        = region,
        scale         = SCALE_M,
        maxPixels     = 1e13,
        formatOptions = {"cloudOptimized": True}
    ).start()

    print(f"  [OK] started -> Drive/{DRIVE_FOLDER}/{file_prefix}.tif")
    launched += 1
    time.sleep(SLEEP_SEC)

print(f"\n[DONE] started {launched} export tasks. Check GEE Tasks panel.")



[1/30] S1A_EW_GRDM_1SDH_20150329T132105_20150329T132216_005247_006A0A_8BDC
  [OK] started -> Drive/S1_EW_exports/S1A_EW_GRDM_1SDH_20150329T132105_20150329T132216_005247_006A0A_8BDC_EW_HH_HV_angle_int16x100.tif

[2/30] S1A_EW_GRDM_1SDH_20150324T145042_20150324T145142_005175_00686C_D18F
  [OK] started -> Drive/S1_EW_exports/S1A_EW_GRDM_1SDH_20150324T145042_20150324T145142_005175_00686C_D18F_EW_HH_HV_angle_int16x100.tif

[3/30] S1A_EW_GRDM_1SDH_20150319T144329_20150319T144410_005102_0066AB_0D49
  [OK] started -> Drive/S1_EW_exports/S1A_EW_GRDM_1SDH_20150319T144329_20150319T144410_005102_0066AB_0D49_EW_HH_HV_angle_int16x100.tif

[4/30] S1A_EW_GRDM_1SDH_20150312T145034_20150312T145134_005000_006442_905F
  [OK] started -> Drive/S1_EW_exports/S1A_EW_GRDM_1SDH_20150312T145034_20150312T145134_005000_006442_905F_EW_HH_HV_angle_int16x100.tif

[5/30] S1A_EW_GRDM_1SDH_20150305T181522_20150305T181625_004900_0061C5_7D65
  [OK] started -> Drive/S1_EW_exports/S1A_EW_GRDM_1SDH_20150305T181522_20150305T

KeyboardInterrupt: 